# Retail Data Cleaning

This notebook cleans the raw retail transaction file in small, easy-to-run steps.

**Input:** `../data/retail_dirty_dataset_2000.csv`  
**Output:** `../data/retail_cleaned_dataset.csv`

The cleaning rules are written in the code comments so each decision is easy to understand and change.

## 1. Import libraries and set file paths

Run this cell first. `Path` builds file locations relative to this notebook, so the code works without hard-coded computer-specific paths.

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd

# Find the project folder from either a project-root kernel or a notebooks-folder kernel.
PROJECT_DIR = Path.cwd()
if PROJECT_DIR.name.lower() == "notebooks":
    PROJECT_DIR = PROJECT_DIR.parent
elif not (PROJECT_DIR / "data").is_dir() and (PROJECT_DIR.parent / "data").is_dir():
    PROJECT_DIR = PROJECT_DIR.parent

INPUT_FILE = PROJECT_DIR / "data" / "retail_dirty_dataset_2000.csv"
OUTPUT_FILE = PROJECT_DIR / "data" / "Retail_Cleaned.csv"
LEGACY_OUTPUT_FILE = PROJECT_DIR / "data" / "retail_cleaned_dataset.csv"
ENCODED_OUTPUT_FILE = PROJECT_DIR / "data" / "Retail_Cleaned_Encoded.csv"

print(f"Input file:  {INPUT_FILE}")
print(f"Output file: {OUTPUT_FILE}")

## 2. Load and inspect the dirty data

This cell does not change anything. It gives us a baseline: row count, column names, data types, missing values, and exact duplicate rows.

In [ ]:
# Read every column as text first. This preserves dirty values such as "wrong", "five", and "not-a-date"
# so we can handle them deliberately instead of losing information during import.
df = pd.read_csv(INPUT_FILE, dtype="string")
rows_before = len(df)

print(f"Rows: {df.shape[0]:,}")
print(f"Columns: {df.shape[1]}")
print("\nColumn names:")
print(df.columns.tolist())

print("\nData types before cleaning:")
print(df.dtypes.to_string())

print("\nMissing-like values before cleaning:")
missing_markers = ["", "null", "none", "nan", "na", "n/a", "unknown", "?", "wrong", "not-a-date"]
missing_like_counts = df.apply(
    lambda column: column.astype("string").str.strip().str.lower().isin(missing_markers).sum()
)
print(missing_like_counts.to_string())

print(f"Exact duplicate rows before cleaning: {df.duplicated().sum():,}")

## 3. Standardize text and missing values

The raw file uses many spellings for missing data and inconsistent capitalization. We first convert those markers to real `NaN` values, then standardize the categorical columns.

In [ ]:
# These values mean "missing" in this dataset. Converting them to NaN lets pandas handle them consistently.
MISSING_MARKERS = {
    "", "null", "none", "nan", "na", "n/a", "unknown",
    "?", "wrong", "not-a-date"
}

for column in df.columns:
    # `astype("string")` keeps missing values safe while `.str.strip()` removes extra spaces.
    cleaned_text = df[column].astype("string").str.strip()
    is_missing = cleaned_text.str.lower().isin(MISSING_MARKERS)
    df[column] = cleaned_text.mask(is_missing, pd.NA)

# Categories are normalized so values like "grocery", "Groceries ", and "GROCERIES"
# become one consistent category.
df["Gender"] = df["Gender"].str.strip().str.title()
df["Gender"] = df["Gender"].replace({"M": "Male", "F": "Female", "O": "Other"})
df["Gender"] = df["Gender"].where(df["Gender"].isin(["Male", "Female", "Other"]))
df["Gender"] = df["Gender"].fillna("Other")

for column in ["City", "ProductCategory"]:
    df[column] = df[column].str.strip().str.title()

df["ProductCategory"] = df["ProductCategory"].replace({"Grocery": "Groceries"})
df["PaymentMode"] = df["PaymentMode"].str.strip().str.lower()
df["PaymentMode"] = df["PaymentMode"].replace(
    {"upi": "UPI", "card": "Card", "cash": "Cash", "wallet": "Wallet", "e-wallet": "E-wallet"}
)
df["PaymentMode"] = df["PaymentMode"].where(
    df["PaymentMode"].isin(["UPI", "Card", "Cash", "Wallet", "E-wallet"])
)

# Product category is essential for analysis, so rows without it are removed later.
# City and payment mode are less critical, so their missing values are filled with the mode.
for column in ["City", "PaymentMode"]:
    mode_value = df[column].mode(dropna=True)
    if not mode_value.empty:
        df[column] = df[column].fillna(mode_value.iloc[0])

print("Gender values:", sorted(df["Gender"].dropna().unique().tolist()))
print("Payment values:", sorted(df["PaymentMode"].dropna().unique().tolist()))
print("Product categories:", sorted(df["ProductCategory"].dropna().unique().tolist()))

## 4. Clean numeric columns and dates

`pd.to_numeric(..., errors="coerce")` changes invalid values to `NaN`. We then fill numeric gaps with transparent median rules. The transaction total is recalculated from quantity and price so it cannot disagree with its parts.

In [ ]:
# Convert columns that should contain numbers. Invalid text becomes NaN instead of crashing the notebook.
for column in ["Age", "Quantity", "Price", "TotalAmount"]:
    df[column] = pd.to_numeric(df[column], errors="coerce")

# Age must be realistic for this customer dataset. Invalid ages are treated as missing.
valid_age = df["Age"].between(18, 100)
age_median = df.loc[valid_age, "Age"].median()
df.loc[~valid_age, "Age"] = np.nan
df["Age"] = df["Age"].fillna(age_median).round().astype("int64")

# Quantity must be a positive whole number. The median is a robust replacement for bad values.
quantity_median = df.loc[df["Quantity"] > 0, "Quantity"].median()
quantity_fill = max(1, int(round(quantity_median)))
df.loc[df["Quantity"] <= 0, "Quantity"] = np.nan
df["Quantity"] = df["Quantity"].fillna(quantity_fill).round().astype("int64")

# Prices must be positive. Negative prices, zero, and text values are replaced by the median valid price.
price_median = df.loc[df["Price"] > 0, "Price"].median()
df.loc[df["Price"] <= 0, "Price"] = np.nan
df["Price"] = df["Price"].fillna(price_median).round(2)

# Recalculate the amount instead of trusting the dirty TotalAmount column.
df["TotalAmount"] = (df["Quantity"] * df["Price"]).round(2)

# Parse each date format separately because the raw file mixes ISO, slash, day-first,
# month-first, and two-digit-year formats. Invalid dates remain NaT and are removed.
date_formats = [
    "%Y-%m-%d", "%Y/%m/%d", "%d/%m/%Y", "%m-%d-%Y",
    "%d-%m-%Y", "%d-%m-%y", "%m-%d-%y",
]
parsed_dates = pd.Series(pd.NaT, index=df.index, dtype="datetime64[ns]")
for date_format in date_formats:
    parsed_dates = parsed_dates.fillna(
        pd.to_datetime(df["PurchaseDate"], format=date_format, errors="coerce")
    )
df["PurchaseDate"] = parsed_dates
df = df.dropna(subset=["PurchaseDate"])
df["PurchaseDate"] = df["PurchaseDate"].dt.strftime("%Y-%m-%d")

print(f"Age median used: {age_median:.1f}")
print(f"Quantity replacement used: {quantity_fill}")
print(f"Price median used: {price_median:.2f}")
print(f"Rows after valid-date filtering: {len(df):,}")

## 5. Remove unusable rows and create useful features

Transaction and customer IDs identify records, so rows without either ID are removed. Exact duplicate rows are removed after normalization. Finally, we add simple analysis-ready columns.

In [ ]:
# Remove rows missing fields that identify a transaction or define its product category.
df = df.dropna(subset=["TransactionID", "CustomerID", "ProductCategory"])

# Remove exact duplicates only. We do not remove repeated IDs automatically because one customer
# can make multiple purchases, and a repeated transaction ID needs business confirmation.
df = df.drop_duplicates().reset_index(drop=True)

# Create features that are useful for later charts and analysis.
df["PurchaseDate"] = pd.to_datetime(df["PurchaseDate"])
df["Month"] = df["PurchaseDate"].dt.to_period("M").astype("string")
df["DayOfWeek"] = df["PurchaseDate"].dt.day_name()
df["AgeGroup"] = pd.cut(
    df["Age"],
    bins=[17, 25, 40, 60, 100],
    labels=["18-25", "26-40", "41-60", "60+"],
)
df["PurchaseDate"] = df["PurchaseDate"].dt.strftime("%Y-%m-%d")

print(f"Rows after ID/category filtering and duplicate removal: {len(df):,}")
print(df.head().to_string(index=False))

## 6. Save the cleaned CSV

This cell writes a new file and leaves the original dirty CSV unchanged. The output is ready for analysis or visualization.

In [ ]:
# Save the readable cleaned dataset required by the assignment.
df["AgeGroup"] = df["AgeGroup"].astype("string")
df.to_csv(OUTPUT_FILE, index=False)

# Keep the earlier lowercase filename as a compatibility copy for existing project references.
df.to_csv(LEGACY_OUTPUT_FILE, index=False)

# One-hot encode categorical columns for analysis or machine-learning models.
categorical_columns = ["Gender", "City", "ProductCategory", "PaymentMode", "DayOfWeek", "AgeGroup"]
encoded_df = pd.get_dummies(df, columns=categorical_columns, dtype=int)

# Min-max normalization puts selected numeric values on a 0-to-1 scale.
numeric_columns = ["Age", "Quantity", "Price", "TotalAmount"]
for column in numeric_columns:
    minimum = encoded_df[column].min()
    maximum = encoded_df[column].max()
    if maximum != minimum:
        encoded_df[f"{column}_Normalized"] = (encoded_df[column] - minimum) / (maximum - minimum)
    else:
        encoded_df[f"{column}_Normalized"] = 0.0
encoded_df.to_csv(ENCODED_OUTPUT_FILE, index=False)

print(f"Saved readable cleaned file: {OUTPUT_FILE}")
print(f"Saved encoded analysis file: {ENCODED_OUTPUT_FILE}")
print(f"Cleaned rows: {len(df):,}")
print(f"Readable columns: {len(df.columns)}")
print(f"Encoded columns: {len(encoded_df.columns)}")

## 7. Verify the output

A cleaning script should prove that its output meets the rules. This final cell reloads the new CSV and checks missing values, numeric ranges, dates, totals, and duplicates.

In [ ]:
cleaned_check = pd.read_csv(OUTPUT_FILE)

# These checks are the promised data-quality rules.
assert cleaned_check["TransactionID"].notna().all()
assert cleaned_check["CustomerID"].notna().all()
assert cleaned_check["ProductCategory"].notna().all()
assert cleaned_check["Age"].between(18, 100).all()
assert (cleaned_check["Quantity"] > 0).all()
assert (cleaned_check["Price"] > 0).all()
assert pd.to_datetime(cleaned_check["PurchaseDate"], errors="coerce").notna().all()
assert not cleaned_check.duplicated().any()

expected_total = (cleaned_check["Quantity"] * cleaned_check["Price"]).round(2)
assert np.allclose(cleaned_check["TotalAmount"], expected_total)

print("All validation checks passed.")
print(f"Rows removed from the raw file: {rows_before - len(cleaned_check):,}")
print(f"Remaining missing values: {int(cleaned_check.isna().sum().sum()):,}")
print(cleaned_check.head().to_string(index=False))